In [5]:
from pathlib import Path
import datetime
import json
import subprocess
import sys

PIPE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py"

# GDAL-enabled python (usually this is fine on EASI)
PYTHON = sys.executable

# ---- REQUIRED PARAMS ----
TILE = "089_084"            # PPP_RRR
START_DATE = "20230728"     # YYYYMMDD
END_DATE   = "20240831"     # YYYYMMDD

# EASI roots (typical)
SR_ROOT = "/home/jovyan/scratch/eds/tiles"
FC_ROOT = "/home/jovyan/scratch/eds/tiles"

# Where pipeline writes outputs (choose a stable location)
OUT_ROOT = "/home/jovyan/work-easi-eds/data/compat/files"

# ---- OPTIONAL KNOBS (good defaults) ----
TIMESERIES_SOURCE = "fc"    # "fc" is typical in your workflow
FC_ONLY_CLR = True          # prefer *_clr variants where available
FORCE_COMPAT = False        # set True if you want to rebuild compat outputs
DRY_RUN = False             # set True to preview commands without running


In [6]:
cmd = [
    PYTHON, PIPE,
    "--tile", TILE,
    "--start-date", START_DATE,
    "--end-date", END_DATE,
    "--sr-root", SR_ROOT,
    "--fc-root", FC_ROOT,
    "--out-root", OUT_ROOT,
    "--timeseries-source", TIMESERIES_SOURCE,
]

if FC_ONLY_CLR:
    cmd.append("--fc-only-clr")

if FORCE_COMPAT:
    cmd.append("--force-compat")

if DRY_RUN:
    cmd.append("--dry-run")

print("Command to be executed:\n")
print(" ".join(cmd))


Command to be executed:

/home/jovyan/venvs/eds-slats/bin/python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py --tile 089_084 --start-date 20230728 --end-date 20240831 --sr-root /home/jovyan/scratch/eds/tiles --fc-root /home/jovyan/scratch/eds/tiles --out-root /home/jovyan/work-easi-eds/data/compat/files --timeseries-source fc --fc-only-clr


In [7]:
t0 = datetime.datetime.utcnow()

proc = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
)

t1 = datetime.datetime.utcnow()

# Put logs in a sensible, discoverable place:
# use a dedicated run-logs folder under OUT_ROOT
run_logs_dir = Path(OUT_ROOT) / "_run_logs"
run_logs_dir.mkdir(parents=True, exist_ok=True)

log_path = run_logs_dir / f"eds_master_{TILE}_d{START_DATE}_{END_DATE}_{t0.strftime('%Y%m%dT%H%M%SZ')}.json"

run_log = {
    "pipeline": PIPE,
    "python": PYTHON,
    "tile": TILE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "sr_root": SR_ROOT,
    "fc_root": FC_ROOT,
    "out_root": OUT_ROOT,
    "timeseries_source": TIMESERIES_SOURCE,
    "fc_only_clr": FC_ONLY_CLR,
    "force_compat": FORCE_COMPAT,
    "dry_run": DRY_RUN,
    "start_time_utc": t0.isoformat() + "Z",
    "end_time_utc": t1.isoformat() + "Z",
    "duration_sec": (t1 - t0).total_seconds(),
    "command": cmd,
    "returncode": proc.returncode,
    "stdout_tail": proc.stdout[-8000:],  # keep it readable
    "stderr_tail": proc.stderr[-8000:],
}

with open(log_path, "w") as f:
    json.dump(run_log, f, indent=2)

print(f"Saved run log: {log_path}")
print(f"Return code: {proc.returncode}")


/tmp/ipykernel_8061/3809589843.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  t0 = datetime.datetime.utcnow()


Saved run log: /home/jovyan/work-easi-eds/data/compat/files/_run_logs/eds_master_089_084_d20230728_20240831_20260104T231343Z.json
Return code: 1


/tmp/ipykernel_8061/3809589843.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  t1 = datetime.datetime.utcnow()


In [8]:
print("----- STDOUT (tail) -----")
print(proc.stdout[-4000:])

print("----- STDERR (tail) -----")
print(proc.stderr[-4000:])

if proc.returncode != 0:
    raise RuntimeError(f"EDS pipeline failed (see {log_path})")

print("EDS pipeline completed successfully.")


----- STDOUT (tail) -----
NGE]  /home/jovyan/scratch/eds/tiles/p089r084/fc/2017/201704/galsfc3_p089r084_20170422_fcm6_clr.tif
  20170508 [OUT-OF-RANGE]  /home/jovyan/scratch/eds/tiles/p089r084/fc/2017/201705/galsfc3_p089r084_20170508_fcm6_clr.tif
  20170524 [OUT-OF-RANGE]  /home/jovyan/scratch/eds/tiles/p089r084/fc/2017/201705/galsfc3_p089r084_20170524_fcm6_clr.tif
  20170625 [OUT-OF-RANGE]  /home/jovyan/scratch/eds/tiles/p089r084/fc/2017/201706/galsfc3_p089r084_20170625_fcm6_clr.tif
  20170711 [OUT-OF-RANGE]  /home/jovyan/scratch/eds/tiles/p089r084/fc/2017/201707/galsfc3_p089r084_20170711_fcm6_clr.tif
  20170727 [OUT-OF-RANGE]  /home/jovyan/scratch/eds/tiles/p089r084/fc/2017/201707/galsfc3_p089r084_20170727_fcm6_clr.tif
  20170812 [OUT-OF-RANGE]  /home/jovyan/scratch/eds/tiles/p089r084/fc/2017/201708/galsfc3_p089r084_20170812_fcm6_clr.tif
  20170913 [OUT-OF-RANGE]  /home/jovyan/scratch/eds/tiles/p089r084/fc/2017/201709/galsfc3_p089r084_20170913_fcm6_clr.tif
  20171015 [OUT-OF-RANGE]  

RuntimeError: EDS pipeline failed (see /home/jovyan/work-easi-eds/data/compat/files/_run_logs/eds_master_089_084_d20230728_20240831_20260104T231343Z.json)